# model-save-state-dict — worked example 3: state_dict save then load_state_dict restore

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `model-save-state-dict`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

`t.save(model.state_dict(), path)` persists only the parameter tensors (a plain dict), not the model class. To restore, you reconstruct the architecture and call `model.load_state_dict(t.load(path))`, which copies the saved tensors into the live parameters in place.

## Worked solution

We do a full save/restore round-trip into a freshly built model.

1. **Train-ish state.** We set the source model's weights to a recognizable value so we can confirm the restore worked.
2. **Save the state_dict.** `t.save(src.state_dict(), path)` writes the parameter dict only — portable across runs and machines.
3. **Rebuild architecture.** A new `nn.Linear` with random init has different weights; we will overwrite them.
4. **load_state_dict.** `dst.load_state_dict(t.load(path, weights_only=True))` copies the saved tensors into `dst`'s parameters, so `dst` now matches `src` exactly.

The demo prints that the restored model's weights equal the source's after the round-trip.

In [ ]:
import torch as t
import torch.nn as nn
import tempfile, os

t.manual_seed(2)

src = nn.Linear(4, 2)
with t.no_grad():
    src.weight.fill_(3.5)
    src.bias.fill_(-1.0)

tmp = tempfile.mkdtemp()
path = os.path.join(tmp, 'sd.pt')
t.save(src.state_dict(), path)

dst = nn.Linear(4, 2)        # fresh random init
before_match = t.allclose(dst.weight, src.weight)
dst.load_state_dict(t.load(path, weights_only=True))
after_match = t.allclose(dst.weight, src.weight) and t.allclose(dst.bias, src.bias)
print('matched before restore:', before_match)
print('matched after restore:', after_match)